# Early-Fusion MIL (CT PNG slices + RNA) — Patient-level, Stratified, Robust Training
**Fixes included:** correct patient-level splits, label alignment checks, proper BCEWithLogitsLoss (no pre-sigmoid), train-fold `pos_weight`, per-patient z-score, frozen backbone warmup, RNA standardization on train-only, early stopping, reasonable LR schedule, and full AUROC/AUPRC/confusion evaluation.

_Generated: 2025-11-07 18:18 UTC_


> **Before you run:**  
> 1) Set `IMAGE_ROOT` to your PNG folder: `D:\\research\\Thesis_data\\TCGA-LUAD\\TCGA-LUAD png`  
> 2) Point `CSV_PATH` to your CSV that contains `patient_id`, `label` (0/1), and RNA feature columns (anything not `patient_id`/`label` will be treated as RNA features).  
> 3) Ensure you have Python packages: `torch`, `torchvision`, `pandas`, `numpy`, `scikit-learn`, `tqdm`, `matplotlib`.
>
> The notebook will:
> - Validate label alignment and report any missing images/labels.
> - Do **patient-level stratified split** to avoid leakage.
> - Compute **pos_weight on the train fold** only.
> - Use **BCEWithLogitsLoss** (apply sigmoid **only** at evaluation).
> - Normalize slices with a simple **per-bag z-score** (approx) each batch.
> - **Freeze** the ResNet backbone first; train attention + fusion MLP, then **unfreeze last block**.
> - **Standardize RNA** with `StandardScaler` fit on **train** fold only; apply to val/test.
> - Use small LR, cosine schedule, **early stop** on best **val AUROC** and save a checkpoint.
> - Report **AUROC, AUPRC, accuracy, confusion matrix** at a threshold picked on the **val** set.


In [ ]:

# ========= Config =========
from pathlib import Path

# !!! EDIT THESE TWO PATHS FOR YOUR MACHINE !!!
IMAGE_ROOT = r"D:\research\Thesis_data\TCGA-LUAD\TCGA-LUAD png"   # <-- your PNG directory
CSV_PATH   = r"D:\research\Thesis_data\TCGA-LUAD\selected_clean_patients_with_labels.csv"  # patient_id,label, RNA columns...

# Training hyperparams
SEED = 42
NUM_EPOCHS = 40
BATCH_SIZE = 1           # One patient bag per batch (MIL)
K_SLICES = 24            # number of slices sampled per patient per epoch
IMG_SIZE = 224           # feed to resnet18
BACKBONE = "resnet18"    # torchvision backbone
LR_WARMUP = 1e-4         # warmup LR when backbone frozen
LR_FINETUNE = 5e-5       # LR when last block unfrozen
WARMUP_EPOCHS = 8
PATIENCE = 8             # early stopping patience (epochs without improvement)
NUM_WORKERS = 4

# Logging/checkpoints
OUT_DIR = Path("./runs_mil_fusion")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

# ========= Imports & seed =========
import os, random, math, time, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms
from PIL import Image

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve, roc_curve

from tqdm import tqdm
import matplotlib.pyplot as plt

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:

# ========= Data loading & validation =========
# Assumptions:
# - PNG files live under IMAGE_ROOT with filenames that contain patient_id or follow a folder structure per patient.
#   We support the following typical layouts:
#   1) IMAGE_ROOT/<patient_id>/*.png
#   2) IMAGE_ROOT/*.png with filenames like <patient_id>_something.png

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp"}

def index_images(image_root: str) -> Dict[str, List[Path]]:
    root = Path(image_root)
    assert root.exists(), f"IMAGE_ROOT not found: {root}"
    patient_to_paths = {}

    # Option A: subfolders per patient
    for pdir in root.iterdir():
        if pdir.is_dir():
            pid = pdir.name.strip()
            paths = [p for p in pdir.rglob("*") if p.suffix.lower() in IMG_EXTS]
            if paths:
                patient_to_paths[pid] = sorted(paths)

    # Option B: flat folder with patient_id prefix in filename
    flat_paths = [p for p in root.glob("*") if p.suffix.lower() in IMG_EXTS]
    for p in flat_paths:
        base = p.stem
        # Try split by common separators
        pid = base.split("_")[0].split("-slide")[0]
        if pid:
            patient_to_paths.setdefault(pid, []).append(p)

    # Ensure sorted and unique
    for k,v in list(patient_to_paths.items()):
        uniq = sorted(set(v))
        if not uniq:
            del patient_to_paths[k]
        else:
            patient_to_paths[k] = uniq
    return patient_to_paths

df = pd.read_csv(CSV_PATH)
assert "patient_id" in df.columns, "CSV must include a 'patient_id' column"
assert "label" in df.columns, "CSV must include a 'label' column (0/1)"

# RNA features = all non patient_id/label columns
rna_cols = [c for c in df.columns if c not in ["patient_id","label"]]
assert len(rna_cols) > 0, "CSV must include RNA feature columns besides patient_id/label"

# Cast label to int
df["label"] = df["label"].astype(int)

patient_images = index_images(IMAGE_ROOT)

# Report coverage
have_img = df["patient_id"].isin(patient_images.keys())
missing_img_patients = df.loc[~have_img, "patient_id"].tolist()
print(f"Total patients in CSV: {len(df)}")
print(f"Patients with images: {have_img.sum()}")
if missing_img_patients:
    print("Patients missing images:", missing_img_patients)

# Keep only patients that have images
df = df.loc[have_img].reset_index(drop=True)

# Basic label distribution
print("Label counts:\n", df["label"].value_counts(dropna=False).sort_index())


In [ ]:

# ========= Patient-level stratified split =========
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# We'll just take fold 0 as val for simplicity
train_idx, val_idx = next(iter(skf.split(df["patient_id"], df["label"])))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

print("Train size:", len(train_df), "Val size:", len(val_df))
print("Train label dist:\n", train_df["label"].value_counts().sort_index())
print("Val label dist:\n", val_df["label"].value_counts().sort_index())


In [ ]:

# ========= RNA scaler (fit on train only) =========
rna_scaler = StandardScaler()
rna_scaler.fit(train_df[rna_cols].values)

train_df_std = train_df.copy()
val_df_std   = val_df.copy()
train_df_std[rna_cols] = rna_scaler.transform(train_df[rna_cols].values)
val_df_std[rna_cols]   = rna_scaler.transform(val_df[rna_cols].values)


In [ ]:

# ========= Dataset & transforms =========
to_tensor = transforms.ToTensor()
resize = transforms.Resize((IMG_SIZE, IMG_SIZE))

class PatientBagDataset(Dataset):
    def __init__(self, df_pat: pd.DataFrame, patient_images: Dict[str, List[Path]],
                 rna_cols: List[str], k_slices=K_SLICES, training=True):
        self.df = df_pat.reset_index(drop=True)
        self.patient_images = patient_images
        self.rna_cols = rna_cols
        self.k = k_slices
        self.training = training

    def __len__(self):
        return len(self.df)

    def _sample_indices(self, n: int, k: int) -> List[int]:
        if n <= k:
            # repeat some indices if fewer than K slices
            idxs = list(range(n))
            while len(idxs) < k:
                idxs += idxs[:max(1, k - len(idxs))]
            return idxs[:k]
        # Random offset sampling to avoid position artifacts
        offset = random.randint(0, n-1) if self.training else 0
        step = n / k
        idxs = [int((offset + i * step) % n) for i in range(k)]
        return idxs

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pid = str(row["patient_id"])
        label = int(row["label"])
        rna  = row[self.rna_cols].values.astype(np.float32)

        paths = self.patient_images.get(pid, [])
        assert len(paths) > 0, f"No images for patient {pid}"

        idxs = self._sample_indices(len(paths), self.k)

        # Load & stack slices
        imgs = []
        for j in idxs:
            im = Image.open(paths[j]).convert("L")  # CT-like gray
            im = resize(im)
            t = to_tensor(im)  # [1,H,W], in [0,1]
            imgs.append(t)
        bag = torch.stack(imgs, dim=0)  # [K,1,H,W]

        # Per-bag z-score (approx): compute mean/std over bag pixels
        with torch.no_grad():
            m = bag.mean()
            s = bag.std()
            bag = (bag - m) / (s + 1e-6)

        return bag.float(), torch.from_numpy(rna), torch.tensor(label, dtype=torch.float32), pid

train_ds = PatientBagDataset(train_df_std, patient_images, rna_cols, k_slices=K_SLICES, training=True)
val_ds   = PatientBagDataset(val_df_std,   patient_images, rna_cols, k_slices=K_SLICES, training=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
len(train_ds), len(val_ds)


In [ ]:

# ========= MIL Attention + Fusion Model =========
class AttentionMIL(nn.Module):
    def __init__(self, feat_dim, attn_dim=128):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(feat_dim, attn_dim),
            nn.Tanh(),
            nn.Linear(attn_dim, 1)
        )

    def forward(self, feats):  # feats: [B,K,D]
        # Compute attention weights over K
        a = self.attn(feats).squeeze(-1)  # [B,K]
        w = torch.softmax(a, dim=1)       # [B,K]
        bag_emb = torch.sum(w.unsqueeze(-1) * feats, dim=1)  # [B,D]
        return bag_emb, w

class FusionModel(nn.Module):
    def __init__(self, rna_dim, backbone_name="resnet18", pretrained=True, freeze_backbone=True):
        super().__init__()
        if backbone_name == "resnet18":
            m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
            cnn_dim = m.fc.in_features
            m.fc = nn.Identity()
            self.backbone = m
        else:
            raise NotImplementedError(backbone_name)

        self.freeze_backbone = freeze_backbone
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        self.mil = AttentionMIL(cnn_dim)
        self.fusion = nn.Sequential(
            nn.Linear(cnn_dim + rna_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 1)  # binary logits
        )

    def unfreeze_last_block(self):
        # Unfreeze last block (layer4) for fine-tuning
        for p in self.backbone.layer4.parameters():
            p.requires_grad = True

    def extract_feats(self, x):  # x: [B,K,1,H,W]
        B,K,_,H,W = x.shape
        x = x.view(B*K, 1, H, W).repeat(1,3,1,1)  # to 3ch
        feats = self.backbone(x)  # [B*K, C]
        feats = feats.view(B, K, -1)
        return feats

    def forward(self, bag_imgs, rna):
        feats = self.extract_feats(bag_imgs)  # [B,K,C]
        bag_emb, attn_w = self.mil(feats)    # [B,C], [B,K]
        x = torch.cat([bag_emb, rna], dim=1) # [B, C+R]
        logit = self.fusion(x).squeeze(1)    # [B]
        return logit, attn_w


In [ ]:

# ========= Training utils =========
def compute_pos_weight(labels: List[int]):
    # labels is list of {0,1}
    pos = sum(labels)
    neg = len(labels) - pos
    if pos == 0:
        return torch.tensor(1.0, device=device)
    return torch.tensor(neg / max(1, pos), device=device)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    losses = []
    for bag, rna, y, _ in tqdm(loader, desc="Train", leave=False):
        bag = bag.to(device, non_blocking=True)
        rna = rna.to(device, non_blocking=True)
        y   = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits, _ = model(bag, rna)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses)) if losses else np.nan

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps, pids = [], [], []
    for bag, rna, y, pid in tqdm(loader, desc="Eval", leave=False):
        bag = bag.to(device, non_blocking=True)
        rna = rna.to(device, non_blocking=True)
        y   = y.to(device, non_blocking=True)

        logits, _ = model(bag, rna)
        prob = torch.sigmoid(logits)
        ys.extend(y.cpu().numpy().tolist())
        ps.extend(prob.cpu().numpy().tolist())
        pids.extend(list(pid))

    ys = np.array(ys).astype(int)
    ps = np.array(ps).astype(float)
    if len(np.unique(ys)) == 1:
        auroc = float("nan")
        auprc = float("nan")
    else:
        auroc = roc_auc_score(ys, ps)
        auprc = average_precision_score(ys, ps)

    # Choose threshold on validation set by maximizing Youden's J (tpr - fpr)
    fpr, tpr, thr = roc_curve(ys, ps)
    j = tpr - fpr
    best_idx = int(np.argmax(j))
    best_thr = float(thr[best_idx]) if len(thr)>0 else 0.5

    preds = (ps >= best_thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(ys, preds).ravel().tolist() if len(np.unique(preds))>1 else (0,0,0,0)
    acc = (tp + tn) / max(1, (tp+tn+fp+fn))

    metrics = dict(
        AUROC=auroc, AUPRC=auprc, ACC=acc, THR=best_thr,
        TP=tp, TN=tn, FP=fp, FN=fn
    )
    return metrics, ys, ps, pids


In [ ]:

# ========= Train loop with warmup freeze + fine-tune last block =========
rna_dim = len(rna_cols)
model = FusionModel(rna_dim=rna_dim, backbone_name=BACKBONE, pretrained=True, freeze_backbone=True).to(device)

# Criterion with pos_weight computed on TRAIN ONLY
train_labels = train_df_std["label"].astype(int).tolist()
pos_w = compute_pos_weight(train_labels)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

# Optimizers
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_WARMUP, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=WARMUP_EPOCHS)

best_auc = -1
best_path = str((OUT_DIR / "best_model.pt").resolve())
history = []

epochs = NUM_EPOCHS
no_improve = 0

for epoch in range(1, epochs+1):
    phase = "warmup" if epoch <= WARMUP_EPOCHS else "finetune"
    # On the transition, unfreeze last block and reinit optimizer with smaller LR
    if epoch == WARMUP_EPOCHS + 1:
        model.unfreeze_last_block()
        optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINETUNE, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs - WARMUP_EPOCHS))

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    scheduler.step()

    metrics, ys, ps, pids = evaluate(model, val_loader)
    auroc = metrics["AUROC"]
    auprc = metrics["AUPRC"]
    acc   = metrics["ACC"]

    history.append(dict(epoch=epoch, phase=phase, train_loss=train_loss, **metrics))
    print(f"Epoch {epoch:02d} [{phase}]  loss={train_loss:.4f}  val_AUROC={auroc:.4f}  val_AUPRC={auprc:.4f}  acc={acc:.4f}")

    # Early stopping on AUROC
    score = auroc if not math.isnan(auroc) else -1
    if score > best_auc:
        best_auc = score
        no_improve = 0
        torch.save(dict(model_state=model.state_dict(),
                        rna_cols=rna_cols,
                        scaler_mean=rna_scaler.mean_.tolist(),
                        scaler_scale=rna_scaler.scale_.tolist(),
                        config=dict(K_SLICES=K_SLICES, IMG_SIZE=IMG_SIZE)), best_path)
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break

print("Best AUROC:", best_auc, "Checkpoint:", best_path)
pd.DataFrame(history).to_csv(OUT_DIR/"history.csv", index=False)


In [ ]:

# ========= Plot curves & show metrics from last eval =========
hist = pd.read_csv(OUT_DIR/"history.csv")
display(hist.tail())

plt.figure()
plt.plot(hist["epoch"], hist["train_loss"], label="train_loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training loss"); plt.legend(); plt.show()

plt.figure()
plt.plot(hist["epoch"], hist["AUROC"], label="val_AUROC")
plt.plot(hist["epoch"], hist["AUPRC"], label="val_AUPRC")
plt.xlabel("Epoch"); plt.ylabel("Score"); plt.title("Validation AUROC/AUPRC"); plt.legend(); plt.show()

print("Last epoch metrics:")
print(hist.iloc[-1][["AUROC","AUPRC","ACC","THR","TP","TN","FP","FN"]])


In [ ]:

# ========= Sanity checks for label alignment =========
# Verify that labels used during training match original CSV for the patients seen
m_train = train_df[["patient_id","label"]].merge(train_df_std[["patient_id"]], on="patient_id", how="inner")
m_val   = val_df[["patient_id","label"]].merge(val_df_std[["patient_id"]], on="patient_id", how="inner")
print("Train patients:", len(m_train), "Val patients:", len(m_val))
print("Train label balance:\n", m_train["label"].value_counts().sort_index())
print("Val label balance:\n", m_val["label"].value_counts().sort_index())


In [ ]:

# ========= How to load and run inference later =========
# Example inference snippet (after training), using the saved scaler and model weights.
# You can adapt this to create a separate inference script.

def load_model_for_inference(ckpt_path: str, rna_dim: int):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model = FusionModel(rna_dim=rna_dim, backbone_name=BACKBONE, pretrained=False, freeze_backbone=False)
    model.load_state_dict(ckpt["model_state"])
    model.eval().to(device)
    scaler_mean = np.array(ckpt["scaler_mean"], dtype=np.float32)
    scaler_scale = np.array(ckpt["scaler_scale"], dtype=np.float32)
    return model, scaler_mean, scaler_scale

# Example (commented):
# model_inf, m, s = load_model_for_inference(best_path, rna_dim=len(rna_cols))
# ... then construct a PatientBagDataset with the *same* K_SLICES/IMG_SIZE and apply (x - m)/s to RNA before forward pass.
